# Plotting libraries — what tuning can and cannot reach

Each section uses a different library. Two of them tune; one deliberately does not.


In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

np.random.seed(5)


## Shared parameters


In [ ]:
ROWS = 600
SPREAD = 1.0
BINS = 24
ALPHA = 0.75
COLOR = 'teal'
FILL = 0.0


In [ ]:
raw = pd.DataFrame({
    'x': np.random.normal(0, SPREAD, ROWS),
    'y': np.random.normal(2, SPREAD, ROWS),
})
raw.loc[raw.sample(frac=0.1, random_state=1).index, 'x'] = np.nan
print(raw.shape, '|', int(raw['x'].isna().sum()), 'gaps')


## A. pandas — a knob reached through `inplace=True`

`FILL` is never read by the plot cell. It reaches the histogram only through the
in-place call below, which is the case a plain assignment scan cannot see.


In [ ]:
raw.fillna(FILL, inplace=True)


In [ ]:
raw['x'].plot(kind='hist', bins=BINS, alpha=ALPHA, color=COLOR)
plt.title(f'{ROWS} rows, spread={SPREAD}')
plt.show()


## B. seaborn — the same knobs through a wrapper


In [ ]:
sns.histplot(data=raw, x='x', bins=BINS, alpha=ALPHA, color=COLOR)
plt.show()


## C. plotly — discovered, but NOT tunable

The scan finds `POINTS` and `OPACITY` perfectly well. The Tune button will not
appear, because plotly emits `application/vnd.plotly.v1+json` rather than an
image, and the button is gated on an image output. This cell is here so that
limitation is visible rather than surprising.


In [ ]:
POINTS = 300
OPACITY = 0.6


In [ ]:
import plotly.express as px
fig = px.scatter(raw.head(POINTS), x='x', y='y', opacity=OPACITY)
fig


## D. A matplotlib control, for comparison

Same data, same knobs, but an image output — so this one tunes.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(raw['x'], bins=BINS, alpha=ALPHA, color=COLOR)
ax.hist(raw['y'], bins=BINS, alpha=ALPHA, color='indianred')
plt.show()
